# 06 Merge, Evaluate, Plot

            Merge local/Kaggle JSONL records, compute summaries, and save matplotlib figures.

In [ ]:
from pathlib import Path
import json
import os
import re
import string

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

DEFAULT_INPUT_DIRS = [
    "qwen2p5-1.5b",
    "qwen2p5-3b",
    "qwen2p5-7b",
]
DEFAULT_RECORD_GLOBS = [
    "records_experiment_*.jsonl",
    "records_smoke_qwen2p5_1p5B_real_six_conditions_v4_shard*.jsonl",
    "*.jsonl",
]


def parse_env_list(name, default_values):
    raw = os.getenv(name, "").strip()
    if not raw:
        return list(default_values)
    return [value.strip() for value in raw.split(";") if value.strip()]


INPUT_DIRS = [Path(value) for value in parse_env_list("MERGE_INPUT_DIRS", DEFAULT_INPUT_DIRS)]
OUTPUT_DIR = Path(
    os.getenv(
        "MERGE_OUTPUT_DIR",
        "merged_outputs" if not Path("/kaggle/working").exists() else "/kaggle/working/merged_outputs",
    )
)
RECORD_GLOBS = parse_env_list("MERGE_RECORD_GLOBS", DEFAULT_RECORD_GLOBS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def iter_record_paths():
    seen = set()
    for directory in INPUT_DIRS:
        if not directory.exists():
            continue
        for pattern in RECORD_GLOBS:
            paths = list(directory.glob(pattern)) + list(directory.rglob(pattern))
            for path in sorted(paths):
                resolved = path.resolve()
                if resolved not in seen and path.name != "records_all.jsonl":
                    seen.add(resolved)
                    yield path


def read_records(path):
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skipping invalid JSON in {path}:{line_number}")
                continue
            row["_source_file"] = str(path)
            yield row


records = []
for path in iter_record_paths():
    records.extend(read_records(path))

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("No records found. Place JSONL files under qwen2p5-1.5b/, qwen2p5-3b/, or qwen2p5-7b/.")

df = df[df["status"].fillna("success") == "success"].copy()
if "timestamp" in df:
    df = df.sort_values("timestamp")
semantic_key = ["model_name", "axis", "dataset", "condition", "sample_id"]
df = df.drop_duplicates([column for column in semantic_key if column in df.columns], keep="last")

numeric_columns = ["exact_match", "f1", "accuracy", "latency_seconds", "tokens_per_second", "max_memory_allocated_gb"]
for column in numeric_columns:
    if column in df:
        df[column] = pd.to_numeric(df[column], errors="coerce")


ARTICLES = {"a", "an", "the"}


def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [token for token in text.split() if token not in ARTICLES]
    return " ".join(tokens)


def token_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    overlap = 0
    remaining = gold_tokens.copy()
    for token in pred_tokens:
        if token in remaining:
            overlap += 1
            remaining.remove(token)
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def clean_short_qa_answer(answer):
    answer = str(answer).strip()
    answer = re.split(
        r"\.\s+(?:To determine|This|Therefore|Based on|I |The |It |We )",
        answer,
        maxsplit=1,
    )[0]
    answer = re.sub(r"^(the answer is|answer is|answer)\s*:?\s*", "", answer, flags=re.IGNORECASE)
    return answer.strip(" .")


def extract_short_qa_answer(text):
    match = re.search(r"final answer\s*:\s*(.*)", str(text), flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(1)
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    return clean_short_qa_answer(lines[0]) if lines else ""


def extract_numeric(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    if not numbers:
        return ""
    value = numbers[-1]
    return value[:-2] if value.endswith(".0") else value


def extract_json_answer(text):
    try:
        parsed = json.loads(str(text).strip())
    except json.JSONDecodeError:
        return ""
    return str(parsed.get("answer", "")) if isinstance(parsed, dict) else ""


def extract_for_row(row):
    raw = row.get("prediction_raw", "")
    if str(row.get("condition", "")).endswith("_format"):
        parsed = extract_json_answer(raw)
        if parsed:
            return parsed
    if row.get("dataset") == "gsm8k":
        return extract_numeric(raw)
    return extract_short_qa_answer(raw)


def score_for_row(row):
    prediction = row["prediction_normalized"]
    gold = row.get("gold_answer", "")
    if row.get("dataset") == "gsm8k":
        pred_num = extract_numeric(prediction)
        gold_num = extract_numeric(gold)
        score = float(pred_num != "" and pred_num == gold_num)
        return pd.Series({"exact_match": score, "f1": score, "accuracy": score})
    exact_match = float(normalize_text(prediction) == normalize_text(gold))
    return pd.Series({"exact_match": exact_match, "f1": token_f1(prediction, gold), "accuracy": exact_match})


df["prediction_normalized"] = df.apply(extract_for_row, axis=1)
df[["exact_match", "f1", "accuracy"]] = df.apply(score_for_row, axis=1)

print({"records": len(df), "conditions": sorted(df["condition"].dropna().unique().tolist())})

In [ ]:
def metric_for_row(row):
    if row["axis"] == "robustness" and row["dataset"] == "robustness_gsm8k":
        return "accuracy"
    return "accuracy" if row["axis"] == "reasoning" or row["dataset"] == "gsm8k" else "f1"


def row_score(row):
    metric = metric_for_row(row)
    value = row.get(metric)
    return float(value) if pd.notna(value) else 0.0


df["main_score"] = df.apply(row_score, axis=1)

leaderboard = (
    df.groupby(["model_name", "model_size", "axis", "dataset", "condition"], dropna=False)
    .agg(
        n=("unique_key", "count"),
        exact_match_mean=("exact_match", "mean"),
        f1_mean=("f1", "mean"),
        accuracy_mean=("accuracy", "mean"),
        main_score_mean=("main_score", "mean"),
        latency_mean=("latency_seconds", "mean"),
        memory_mean=("max_memory_allocated_gb", "mean"),
        tokens_per_second_mean=("tokens_per_second", "mean"),
    )
    .reset_index()
)

comparison_pairs = [
    ("reasoning_direct", "reasoning_cot", "main_score", "CoT Gain"),
    ("reasoning_direct", "reasoning_oracle_step", "main_score", "Oracle Step Gain"),
    ("context_none", "context_oracle", "main_score", "Oracle Gain"),
    ("context_retrieved", "context_oracle", "main_score", "Retrieval Gap"),
    ("context_oracle_distractor", "context_oracle", "main_score", "Distractor Drop"),
    ("context_oracle_end", "context_oracle", "main_score", "Position Drop"),
    ("knowledge_closed", "knowledge_oracle", "main_score", "Oracle Gain"),
    ("knowledge_distractor", "knowledge_oracle", "main_score", "Distractor Drop"),
    ("robust_original", "robust_para", "main_score", "Paraphrase Sensitivity"),
    ("robust_original", "robust_typo", "main_score", "Typo Sensitivity"),
    ("robust_original", "robust_format", "main_score", "Format Sensitivity"),
]


def matched_comparison(group, base_condition, compare_condition, metric_name):
    columns = ["sample_id", metric_name]
    base = group.loc[group["condition"] == base_condition, columns].dropna(subset=["sample_id"]).copy()
    compare = group.loc[group["condition"] == compare_condition, columns].dropna(subset=["sample_id"]).copy()
    base = base.rename(columns={metric_name: "base_score"}).drop_duplicates("sample_id", keep="last")
    compare = compare.rename(columns={metric_name: "compare_score"}).drop_duplicates("sample_id", keep="last")
    matched = base.merge(compare, on="sample_id", how="inner")
    return base, compare, matched


summary_rows = []
for (model_name, model_size, axis, dataset), group in df.groupby(["model_name", "model_size", "axis", "dataset"], dropna=False):
    for base_condition, compare_condition, metric_name, comparison_name in comparison_pairs:
        if base_condition not in set(group["condition"]) or compare_condition not in set(group["condition"]):
            continue
        base, compare, matched = matched_comparison(group, base_condition, compare_condition, metric_name)
        score_frame = matched if not matched.empty else pd.DataFrame(
            {
                "base_score": [group.loc[group["condition"] == base_condition, metric_name].mean()],
                "compare_score": [group.loc[group["condition"] == compare_condition, metric_name].mean()],
            }
        )
        base_score = score_frame["base_score"].mean()
        compare_score = score_frame["compare_score"].mean()
        if pd.isna(base_score) or pd.isna(compare_score):
            continue
        delta = compare_score - base_score
        summary_rows.append(
            {
                "model_name": model_name,
                "model_size": model_size,
                "axis": axis,
                "dataset": dataset,
                "base_condition": base_condition,
                "compare_condition": compare_condition,
                "n_base": len(base),
                "n_compare": len(compare),
                "n_overlap": len(matched),
                "base_score": base_score,
                "compare_score": compare_score,
                "delta": delta,
                "base_minus_compare": -delta,
                "abs_delta": abs(delta),
                "metric_name": metric_name,
                "comparison_name": comparison_name,
            }
        )

condition_summary = pd.DataFrame(summary_rows)
condition_columns = [
    "model_name",
    "model_size",
    "axis",
    "dataset",
    "base_condition",
    "compare_condition",
    "n_base",
    "n_compare",
    "n_overlap",
    "base_score",
    "compare_score",
    "delta",
    "base_minus_compare",
    "abs_delta",
    "metric_name",
    "comparison_name",
]
condition_summary = condition_summary.reindex(columns=condition_columns)

robustness_rows = []
for (model_name, model_size, dataset), group in df[df["axis"].eq("robustness")].groupby(["model_name", "model_size", "dataset"], dropna=False):
    original = group.loc[group["condition"].eq("robust_original"), "main_score"].mean()
    if pd.isna(original):
        continue
    perturbed = (
        group[group["condition"].isin(["robust_para", "robust_typo", "robust_format"])]
        .groupby("condition", dropna=False)["main_score"]
        .mean()
    )
    if perturbed.empty:
        continue
    best_condition = perturbed.idxmax()
    worst_condition = perturbed.idxmin()
    best_score = float(perturbed.max())
    worst_score = float(perturbed.min())
    robustness_rows.append(
        {
            "model_name": model_name,
            "model_size": model_size,
            "axis": "robustness",
            "dataset": dataset,
            "original_score": float(original),
            "best_perturbed_condition": best_condition,
            "best_perturbed_score": best_score,
            "worst_perturbed_condition": worst_condition,
            "worst_perturbed_score": worst_score,
            "prompt_sensitivity_gap": float((perturbed - original).abs().max()),
            "worst_case_drop": float(original - worst_score),
        }
    )

robustness_summary = pd.DataFrame(robustness_rows)
robustness_columns = [
    "model_name",
    "model_size",
    "axis",
    "dataset",
    "original_score",
    "best_perturbed_condition",
    "best_perturbed_score",
    "worst_perturbed_condition",
    "worst_perturbed_score",
    "prompt_sensitivity_gap",
    "worst_case_drop",
]
robustness_summary = robustness_summary.reindex(columns=robustness_columns)

model_axis_summary = (
    df.groupby(["model_name", "model_size", "axis"], dropna=False)
    .agg(
        n=("unique_key", "count"),
        main_score=("main_score", "mean"),
        latency_mean=("latency_seconds", "mean"),
        memory_mean=("max_memory_allocated_gb", "mean"),
        tokens_per_second_mean=("tokens_per_second", "mean"),
    )
    .reset_index()
)
model_axis_summary["efficiency_score"] = model_axis_summary["main_score"] / model_axis_summary["latency_mean"].clip(lower=1e-9)

paper_main_table = leaderboard[
    [
        "model_name",
        "model_size",
        "axis",
        "dataset",
        "condition",
        "n",
        "main_score_mean",
        "exact_match_mean",
        "f1_mean",
        "accuracy_mean",
        "latency_mean",
        "memory_mean",
    ]
].sort_values(["model_size", "axis", "condition"]).copy()

paper_comparison_table = condition_summary[
    [
        "model_name",
        "model_size",
        "axis",
        "dataset",
        "comparison_name",
        "base_condition",
        "compare_condition",
        "n_overlap",
        "base_score",
        "compare_score",
        "delta",
        "base_minus_compare",
        "abs_delta",
    ]
].sort_values(["model_size", "axis", "comparison_name"]).copy()

paper_axis_table = model_axis_summary.sort_values(["model_size", "axis"]).copy()

efficiency_log = df[
    [
        "unique_key",
        "model_name",
        "model_size",
        "axis",
        "dataset",
        "condition",
        "latency_seconds",
        "tokens_per_second",
        "max_memory_allocated_gb",
        "input_tokens",
        "output_tokens",
    ]
].copy()

In [ ]:
def write_jsonl(path, frame):
    with path.open("w", encoding="utf-8") as f:
        for row in frame.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


write_jsonl(OUTPUT_DIR / "records_all.jsonl", df)
df.to_csv(OUTPUT_DIR / "records_all.csv", index=False)
leaderboard.to_csv(OUTPUT_DIR / "leaderboard.csv", index=False)
condition_summary.to_csv(OUTPUT_DIR / "condition_summary.csv", index=False)
model_axis_summary.to_csv(OUTPUT_DIR / "model_axis_summary.csv", index=False)
robustness_summary.to_csv(OUTPUT_DIR / "robustness_summary.csv", index=False)
efficiency_log.to_csv(OUTPUT_DIR / "efficiency_log.csv", index=False)
paper_main_table.to_csv(OUTPUT_DIR / "paper_main_table.csv", index=False)
paper_comparison_table.to_csv(OUTPUT_DIR / "paper_comparison_table.csv", index=False)
paper_axis_table.to_csv(OUTPUT_DIR / "paper_axis_table.csv", index=False)

print(
    {
        "records_all": len(df),
        "leaderboard_rows": len(leaderboard),
        "condition_summary_rows": len(condition_summary),
        "model_axis_summary_rows": len(model_axis_summary),
        "robustness_summary_rows": len(robustness_summary),
        "output_dir": str(OUTPUT_DIR),
    }
)

In [ ]:
def save_bar(frame, x, y, title, path):
    if frame.empty:
        return
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(frame[x].astype(str), frame[y])
    ax.set_title(title)
    ax.set_ylabel(y)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_heatmap(frame, path):
    if frame.empty:
        return
    matrix = frame.pivot_table(index="model_size", columns="axis", values="main_score", aggfunc="mean")
    if matrix.empty:
        return
    fig, ax = plt.subplots(figsize=(1.8 + len(matrix.columns) * 1.4, 1.6 + len(matrix.index) * 0.7))
    image = ax.imshow(matrix.values, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    for row_index, _ in enumerate(matrix.index):
        for col_index, _ in enumerate(matrix.columns):
            value = matrix.iloc[row_index, col_index]
            label = "" if pd.isna(value) else f"{value:.2f}"
            ax.text(col_index, row_index, label, ha="center", va="center", color="white" if value < 0.5 else "black")
    ax.set_title("Model x Axis Score")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def comparison_frame(name):
    frame = condition_summary[condition_summary["comparison_name"].eq(name)].copy()
    if frame.empty:
        return frame
    frame["label"] = (
        frame["model_size"].astype(str)
        + " / "
        + frame["axis"].astype(str)
        + ": "
        + frame["base_condition"].astype(str)
        + " -> "
        + frame["compare_condition"].astype(str)
    )
    return frame


save_heatmap(model_axis_summary, OUTPUT_DIR / "figure_model_axis_heatmap.png")

oracle = comparison_frame("Oracle Gain")
save_bar(oracle, "label", "delta", "Oracle Gain", OUTPUT_DIR / "figure_oracle_gain.png")

distractor = comparison_frame("Distractor Drop")
save_bar(distractor, "label", "delta", "Distractor Drop", OUTPUT_DIR / "figure_distractor_drop.png")

position = comparison_frame("Position Drop")
save_bar(position, "label", "delta", "Position Drop", OUTPUT_DIR / "figure_position_drop.png")

retrieval = comparison_frame("Retrieval Gap")
save_bar(retrieval, "label", "delta", "Retrieval Gap", OUTPUT_DIR / "figure_retrieval_gap.png")

cot = comparison_frame("CoT Gain")
save_bar(cot, "label", "delta", "CoT Gain", OUTPUT_DIR / "figure_cot_gain.png")

prompt = condition_summary[condition_summary["comparison_name"].str.contains("Sensitivity", na=False)].copy()
if not prompt.empty:
    prompt["label"] = prompt["model_size"].astype(str) + ": " + prompt["base_condition"].astype(str) + " -> " + prompt["compare_condition"].astype(str)
save_bar(prompt, "label", "abs_delta", "Prompt Sensitivity Gap", OUTPUT_DIR / "figure_prompt_sensitivity.png")

if not robustness_summary.empty:
    robustness_plot = robustness_summary.copy()
    robustness_plot["label"] = robustness_plot["model_size"].astype(str) + " / " + robustness_plot["dataset"].astype(str)
    save_bar(robustness_plot, "label", "prompt_sensitivity_gap", "Worst Prompt Sensitivity Gap", OUTPUT_DIR / "figure_robustness_gap.png")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["latency_seconds"], df["main_score"])
ax.set_xlabel("latency_seconds")
ax.set_ylabel("main_score")
ax.set_title("Score vs Latency")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figure_accuracy_latency.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["max_memory_allocated_gb"], df["main_score"])
ax.set_xlabel("max_memory_allocated_gb")
ax.set_ylabel("main_score")
ax.set_title("Score vs Memory")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figure_accuracy_memory.png", dpi=160)
plt.close(fig)